In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
# clone your branch directly
!git clone -b Valeria https://github.com/paolito81/MaskArchitectureAnomaly_CourseProject

%cd /content/MaskArchitectureAnomaly_CourseProject/eomt/

%pip install -r requirements.txt

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 563 (delta 9), reused 20 (delta 5), pack-reused 532 (from 1)
Receiving objects: 100% (563/563), 30.05 MiB | 22.98 MiB/s, done.
Resolving deltas: 100% (272/272), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.8 MB/s eta 0:00:00


In [3]:
cityscapes_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
cityscapes_data_path = "/content/drive/MyDrive/dataset_city_scapes/cityscapes"
cityscapes_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"

In [4]:
coco_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
coco_data_path = "/content/drive/MyDrive/dataset_city_scapes/coco"
coco_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"

In [5]:
coco_copy_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_coco.yaml"

In [6]:
default_root_dir = "/content/drive/MyDrive/eomt_runs"

In [7]:
!git pull
!git branch -a

Already up to date.
* Valeria
  remotes/origin/HEAD -> origin/main
  remotes/origin/Hossein
  remotes/origin/Omar
  remotes/origin/Valeria
  remotes/origin/lora-eomt-test
  remotes/origin/main


In [8]:
import os

os.environ["WANDB_API_KEY"] = (
    "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"
)

In [9]:
fine_tuned_coco_ckpt_path = "/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt"
coco_copy_config_path_eval = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_finetune.yaml"

In [ ]:
!python3 main.py fit \
   -c {coco_copy_config_path_eval} \
   --trainer.max_epochs 1 \
   --model.load_ckpt_class_head False \
   --trainer.default_root_dir {default_root_dir} \
   --trainer.devices 1 \
   --data.batch_size 2 \
   --data.path {cityscapes_data_path} \
   --model.ckpt_path {coco_ckpt_path} 

2026-05-18 20:13:42.618527: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Currently logged in as: valeriaintini2001 (valeriaintini2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.10
wandb: Run data is saved locally in ./wandb/run-20260518_201400-ymmh3ov2
wandb: Run `wandb offline` to turn off sy

In [11]:
import glob

ckpts = sorted(
    glob.glob("/content/drive/MyDrive/eomt_runs/**/*.ckpt", recursive=True)
)

print("Found checkpoints:")
for c in ckpts:
    print(c)

fine_tuned_coco_ckpt_path = ckpts[-1]

print("\nUsing latest checkpoint:")
print(fine_tuned_coco_ckpt_path)

Found checkpoints:
/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt

Using latest checkpoint:
/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt


In [12]:
!python3 eval_shared_miou.py \
  --config {coco_copy_config_path_eval} \
  --ckpt {fine_tuned_coco_ckpt_path} \
  --cityscapes-path {cityscapes_data_path} \
  --device cuda:0 \
  --batch-size 4 \
  --num-workers 2 \
  --no-masked-attn-enabled \
  --wandb-mode online \
  --src-label-space cityscapes

wandb: Currently logged in as: valeriaintini2001 (valeriaintini2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.10
wandb: Run data is saved locally in /content/MaskArchitectureAnomaly_CourseProject/eomt/wandb/run-20260518_214818-7nsh3xme
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run coco_panoptic_eomt_base_640
wandb: ⭐️ View project at https://wandb.ai/valeriaintini2001-politecnico-di-torino/eomt
wandb: 🚀 View run at https://wandb.ai/valeriaintini2001-politecnico-di-torino/eomt/runs/7nsh3xme
model.safetensors: 100% 346M/346M [00:05<00:00, 61.5MB/s]
2026-05-18 21:48:33.414621: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/

In [ ]:
print("Estimated stepping batches:", self.trainer.estimated_stepping_batches)